# Tweet Sentiment Extraction: train on Kaggle

Before running:
1. **Add Input** > Competitions > *Tweet Sentiment Extraction* (gives `/kaggle/input/tweet-sentiment-extraction`).
2. **Settings** > Accelerator: a GPU. **Internet: on** (to clone the code and download RoBERTa).
3. Run all. One seed (5 folds) is one call of `train1`; split the 5 seeds over several sessions if needed.

Each RoBERTa fold model is about 500 MB, so all 25 need about 12 GB of disk.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and pretrained model

In [ ]:
# the fixed code; change the branch to main once it is merged
!git clone -b fix/runnable-pipeline https://github.com/gbirhanu/tse.git /kaggle/working/tse

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('deepset/roberta-base-squad2',
                  local_dir='/kaggle/working/roberta-base-squad2',
                  allow_patterns=['config.json', 'vocab.json', 'merges.txt',
                                  'model.safetensors'])
!ls -la /kaggle/working/roberta-base-squad2

## 2. Settings

To continue an earlier session: add that notebook's output as an input and copy its `work` folder here first, e.g.
`!cp -r /kaggle/input/<previous-output>/work /kaggle/working/` (finished folds are skipped).

In [ ]:
ARGS = ('--data-dir /kaggle/input/tweet-sentiment-extraction '
        '--work-dir /kaggle/working/work '
        '--pretrained /kaggle/working/roberta-base-squad2 '
        '--num-workers 2')
SEEDS = '0 1 2 3 4'   # e.g. '0' in the first session, '1 2' in the next

## 3. Level one: 5 folds per seed (saves models and out-of-fold predictions)

In [ ]:
!cd /kaggle/working/tse && python main.py train1 $ARGS --seeds $SEEDS --fp16

## 4. Level one on test.csv, level two, submission

In [ ]:
!cd /kaggle/working/tse && python main.py test1 $ARGS --seeds $SEEDS --fp16
!cd /kaggle/working/tse && python main.py train2 $ARGS --seeds $SEEDS
!cd /kaggle/working/tse && python main.py predict $ARGS --seeds $SEEDS

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open('/kaggle/working/work/scores.json')), indent=2))
pd.read_csv('/kaggle/working/work/submission.csv').head(10)